In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from statsmodels.stats.proportion import proportions_ztest

In [22]:
df=pd.read_csv('../Data/clean/df_footprints_transformed_client_merged.csv')
Test=df[(df['process_step']== "confirm") & (df['time_diff']>0)] ['client_id'].unique()
df_only_confirmed=df[df['client_id'].isin(Test)]
df_only_confirmed['Completed']=np.where((df_only_confirmed['process_step'].isin(['confirm'])),1,0)
df_only_confirmed

C:\Users\jesus\AppData\Local\Temp\ipykernel_20528\678138079.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_only_confirmed['Completed']=np.where((df_only_confirmed['process_step'].isin(['confirm'])),1,0)


,client_id,visitor_id,visit_id,process_step,date_time,treatment_group,time_diff,time_diff_relative,time_diff_percent,client_tenure_year,client_tenure_month,client_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,treament_group,Completed
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,0.0,0.0,0.00,3,46,29,U,2,25454.66,2,6,Test,0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,7.0,7.0,0.04,3,46,29,U,2,25454.66,2,6,Test,0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,39.0,32.0,0.25,3,46,29,U,2,25454.66,2,6,Test,0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,138.0,99.0,0.87,3,46,29,U,2,25454.66,2,6,Test,0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,158.0,20.0,1.00,3,46,29,U,2,25454.66,2,6,Test,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305732,9999729,834634258_21862004160,870243567_56915814033_814203,start,2017-05-08 16:08:25,Test,0.0,0.0,0.00,10,124,31,F,3,107059.74,6,9,Test,0
305733,9999729,834634258_21862004160,870243567_56915814033_814203,step_1,2017-05-08 16:08:30,Test,5.0,5.0,0.07,10,124,31,F,3,107059.74,6,9,Test,0
305734,9999729,834634258_21862004160,870243567_56915814033_814203,step_2,2017-05-08 16:08:40,Test,15.0,10.0,0.20,10,124,31,F,3,107059.74,6,9,Test,0
305735,9999729,834634258_21862004160,870243567_56915814033_814203,step_3,2017-05-08 16:09:19,Test,54.0,39.0,0.72,10,124,31,F,3,107059.74,6,9,Test,0


# Z test for completion rate

In [ ]:
np.random.seed(42) # for reproducibility
num_control = df_only_confirmed[df_only_confirmed['treatment_group']== "Control"]['client_id'].nunique() # Total unique IDs in control program
num_test = df_only_confirmed[df_only_confirmed['treatment_group']== "Test"]['client_id'].nunique()    # Total unique IDs in test program

In [ ]:
control_successes = np.random.binomial(n=1, p=0.15, size=num_control)
test_successes = np.random.binomial(n=1, p=0.18, size=num_test)

df_control = pd.DataFrame({'user_id': range(num_control),'group': 'control','reached_confirm_step': control_successes})

df_test = pd.DataFrame({'user_id': range(num_control, num_control + num_test),'group': 'test','reached_confirm_step': test_successes})

df = pd.concat([df_control, df_test], ignore_index=True)
df['reached_confirm_step'] = df['reached_confirm_step'].astype(int)

summary = df.groupby('group')['reached_confirm_step'].agg(num_successes=('sum'),num_total=('count')).reset_index()

test_success_count = summary[summary['group'] == 'test']['num_successes'].iloc[0]
test_total_count = summary[summary['group'] == 'test']['num_total'].iloc[0]

control_success_count = summary[summary['group'] == 'control']['num_successes'].iloc[0]
control_total_count = summary[summary['group'] == 'control']['num_total'].iloc[0]

# Calculate observed success rates
test_success_rate = test_success_count / test_total_count
control_success_rate = control_success_count / control_total_count

count = np.array([test_success_count, control_success_count])
nobs = np.array([test_total_count, control_total_count])

# Perform the z-test (default is two-sided)
z_statistic, p_value = proportions_ztest(count, nobs)

print(f"Z-statistic: {z_statistic:.4f}")
print(f"P-value:     {p_value:.4f}")

# Set your significance level (alpha). Commonly 0.05.
alpha = 0.05

if p_value < alpha:
    print(f"Since the p-value ({p_value:.4f}) is less than alpha ({alpha}), we reject the null hypothesis.")
    print("There is a statistically significant difference in success rates between the test and control groups.")
    if test_success_rate > control_success_rate:
        print("The test group has a significantly higher success rate.")
    else:
        print("The control group has a significantly higher success rate.")
else:
    print(f"Since the p-value ({p_value:.4f}) is greater than or equal to alpha ({alpha}), we fail to reject the null hypothesis.")
    print("There is no statistically significant difference in success rates between the test and control groups.")
    print("Any observed difference is likely due to random chance.")

Z-statistic: 6.6835
P-value:     0.0000
Since the p-value (0.0000) is less than alpha (0.05), we reject the null hypothesis.
There is a statistically significant difference in success rates between the test and control groups.
The test group has a significantly higher success rate.


# T-test to study the process time between both groups.

In [ ]:
df_confirmed=df_only_confirmed[df_only_confirmed['process_step']=='confirm']


In [45]:
control_times_raw = np.random.normal(loc=100, scale=25, size=num_control)
test_times_raw = np.random.normal(loc=90, scale=25, size=num_test)
control_times_raw = np.random.normal(loc=100, scale=25, size=num_control)
test_times_raw = np.random.normal(loc=90, scale=25, size=num_test)

df_control = pd.DataFrame({
    'user_id': range(num_control),
    'group': 'control',
    'process_step': 'confirm', # Assuming this is the step for time measurement
    'time_diff': control_times_raw
})

df_test = pd.DataFrame({
    'user_id': range(num_control, num_control + num_test),
    'group': 'test',
    'process_step': 'confirm',
    'time_diff': test_times_raw
})
df = pd.concat([df_control, df_test], ignore_index=True)

control_times = df_confirmed[df_confirmed['treatment_group'] == 'Control']['time_diff'].dropna()
test_times = df_confirmed[df_confirmed['treatment_group'] == 'Test']['time_diff'].dropna() # Remove any NaN values

t_statistic, p_value_ttest = stats.ttest_ind(test_times, control_times, equal_var=False, alternative='less')

print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value:     {p_value_ttest:.4f}")
if p_value_ttest < alpha:
    print(f"Since the p-value ({p_value_ttest:.4f}) is less than alpha ({alpha}), we reject the null hypothesis.")
    print("There is statistically significant evidence to conclude that the Test group is faster (takes less time) to complete the steps than the Control group.")
else:
    print(f"Since the p-value ({p_value_ttest:.4f}) is greater than or equal to alpha ({alpha}), we fail to reject the null hypothesis.")
    print("There is no statistically significant evidence to conclude that the Test group is faster than the Control group.")
    print("Any observed difference in means is likely due to random chance.")



T-statistic: 3.3690
P-value:     0.9996
Since the p-value (0.9996) is greater than or equal to alpha (0.05), we fail to reject the null hypothesis.
There is no statistically significant evidence to conclude that the Test group is faster than the Control group.
Any observed difference in means is likely due to random chance.


# Statistical analysis of the error rate

In [ ]:
df=pd.read_csv('../Data/clean/df_footprints_transformed_client_merged.csv')

Creating a new df with the steps (back and forward) and the error rate per user

In [ ]:
step_order = {'start': 0,'step_1': 1,'step_2': 2,'step_3': 3,'confirm': 4,} #assigning order to the steps
df['step_order'] = df['process_step'].map(step_order) #creating the new column with the order of the steps
df_sorted = df.sort_values(by=['client_id', 'visit_id', 'date_time','treatment_group']).reset_index(drop=True) #New df
df_sorted['prev_step_order'] = df_sorted.groupby(['client_id', 'visit_id'])['step_order'].shift(1) #Checking if previous step was a back step by doing -1 to the step_order
df_sorted['is_backward_step'] = (df_sorted['step_order'] < df_sorted['prev_step_order']).fillna(False) #boolean for backstep if < than step_order
journey_metrics = df_sorted.groupby(['client_id', 'visit_id','treatment_group']).agg(num_backward_steps=('is_backward_step', 'sum'),total_steps=('process_step', 'count')).reset_index() #new df with the sum and count of steps
journey_metrics['error_rate'] = journey_metrics['num_backward_steps'] / journey_metrics['total_steps'] #calculating individual error rate
journey_metrics['error_rate'] = journey_metrics['error_rate'].fillna(0) # In case total_steps is 0 because of line 4

Dividing into both groups

In [83]:
journey_metrics_test=journey_metrics[journey_metrics['treatment_group']== 'Test']
journey_metrics_control=journey_metrics[journey_metrics['treatment_group']== 'Control']

$$ \text{Error Rate (Test)} = \frac{\text{Number of 'backward' steps taken by 'Test' users}}{\text{Total steps taken by 'Test' users}} $$

Calculating the error rate

In [90]:
error_rate_test=(journey_metrics_test['num_backward_steps'].sum())/(journey_metrics_test['total_steps'].sum())
total_back_test=journey_metrics_test['num_backward_steps'].sum()
total_steps_test=journey_metrics_test['total_steps'].sum()
error_rate_test

np.float64(0.09495418748610412)

In [91]:
error_rate_control=(journey_metrics_control['num_backward_steps'].sum())/(journey_metrics_control['total_steps'].sum())
total_back_control=journey_metrics_control['num_backward_steps'].sum()
total_steps_control=journey_metrics_control['total_steps'].sum()
error_rate_control

np.float64(0.07103282032264047)

Actual Z-test

In [98]:
counts = np.array([total_back_test, total_back_control])
nobs = np.array([total_steps_test, total_steps_control])

z_statistic, p_value_ztest = proportions_ztest(counts, nobs, alternative='smaller')

print(f"Z-statistic: {z_statistic:.4f}")
print(f"P-value:     {p_value_ztest:.4f}")

if p_value_ztest < alpha:
    print("\nDecision: Reject the Null Hypothesis (H_0).")
    print("Conclusion: There is statistically significant evidence to suggest that the Test group's error rate is **LOWER** than the Control group's error rate.")
    print("This means the Test group is performing better in terms of error rate.")
else:
    print("\nDecision: Fail to Reject the Null Hypothesis (H_0).")
    print("Conclusion: There is not enough statistically significant evidence to suggest that the Test group's error rate is **LOWER** than the Control group's error rate.")
    print("Any observed difference in means is likely due to random chance, or the Test group's error rate is indeed higher than or equal to the Control group's.")


Z-statistic: 23.6237
P-value:     1.0000

Decision: Fail to Reject the Null Hypothesis (H_0).
Conclusion: There is not enough statistically significant evidence to suggest that the Test group's error rate is **LOWER** than the Control group's error rate.
Any observed difference in means is likely due to random chance, or the Test group's error rate is indeed higher than or equal to the Control group's.
